# Lab 10: Live Gemini Evaluation

**COMPSS 211 | Fall 2026 | Student copy**

Run a small structured-output call and inspect errors after Monday's LLM lecture.

**Date/deadline:** Friday, November 13, 2026

Work through each task and replace the response placeholders with your own answers.

## Scenario

Make one controlled Gemini request, validate the returned record, and compare it with a saved offline case. The comments are synthetic, and the API key stays outside the notebook.

## Goal

- Keep the key in the environment.
- Validate returned labels.
- Compare against a same-data baseline.

## Keep handy

- **Model:** `gemini-3.6-flash`.
- **Offline copy:** The notebook includes a recorded fixture so ordinary validation never spends money.

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
import numpy as np
from IPython.display import display

REQUIRED_PYTHON = "3.12.13"
if sys.version.split()[0] != REQUIRED_PYTHON:
    raise RuntimeError(
        f"This notebook requires Python {REQUIRED_PYTHON}; "
        f"the active kernel is {sys.version.split()[0]}."
    )

COURSE_MARKERS = ("data", "homework", "lab")

def is_course_root(candidate):
    return all((candidate / name).is_dir() for name in COURSE_MARKERS)

def locate_course_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if is_course_root(candidate):
            return candidate

    in_colab = (
        "google.colab" in sys.modules
        or "COLAB_RELEASE_TAG" in os.environ
    )
    if in_colab:
        matches = [
            candidate
            for candidate in Path("/content").iterdir()
            if candidate.is_dir() and is_course_root(candidate)
        ]
        if len(matches) == 1:
            return matches[0]

    raise FileNotFoundError(
        "Course repository not found. Run from a cloned copy of "
        "macss-berkeley/compss-211a. In Colab, clone or upload the "
        "complete repository under /content, then rerun this cell."
    )

COURSE_ROOT = locate_course_root()
DATA_DIR = COURSE_ROOT / "data"
GENERATED_DIR = COURSE_ROOT / "generated"
GENERATED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Python {sys.version.split()[0]} | data={DATA_DIR}")

## Practice: guarded live call and offline fixture

In [ ]:
GEMINI_MODEL = "gemini-3.6-flash"
sample = pd.read_csv(DATA_DIR / "hw4_synthetic_campus_comments.csv").head(3)
fixture = pd.read_csv(DATA_DIR / "hw5_recorded_evaluation_fixture.csv").head(3)

def validate_output(document_id, label):
    if label not in {"transit", "study_space", "accessibility", "food", "safety", "services"}:
        raise ValueError(f"Unexpected label for {document_id}: {label}")
    return {"document_id": document_id, "label": label}

def run_live(sample):
    key = os.getenv("GEMINI_API_KEY")
    if not key:
        raise RuntimeError("Set GEMINI_API_KEY in the environment.")
    from google import genai
    client = genai.Client(api_key=key)
    return [
        client.models.generate_content(
            model=GEMINI_MODEL,
            contents=f"Return one routing label for: {row.text}",
        ).text
        for row in sample.itertuples()
    ]

live_enabled = os.getenv("COMPSS211_LIVE_API", "0") == "1"
raw_outputs = (
    run_live(sample)
    if live_enabled
    else fixture["recorded_llm_label"].tolist()
)
outputs = [
    validate_output(document_id, label)
    for document_id, label in zip(sample["document_id"], raw_outputs)
]
display(pd.DataFrame(outputs))

### Your notes

Before you leave, write down one thing you can now do and one question you still have.

> Write your notes here.

## Exit

The last 10 minutes are reserved for the three-question quiz on Monday's material. Use the lab to practice the ideas before you answer from memory.